In [2]:
pip install psycopg2-binary sqlalchemy

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 6.7 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 5.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ----------------- ---------------------- 1.0/2.4 MB 6.3 MB/s eta 0:00:01
   -------------------------------------- - 2.4/2.4 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 5.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

query = """
SELECT "UserId", "Date", "ProcessName", "DurationSeconds"
FROM "DailySummaries"
ORDER BY "Date"
"""

raw_df = pd.read_sql(query, engine)
raw_df

,UserId,Date,ProcessName,DurationSeconds
0,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,Code.exe,372
1,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,chrome.exe,954
2,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-14,chrome.exe,6
3,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,SearchHost.exe,4
4,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,datagrip64.exe,46
5,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,explorer.exe,1
6,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-14,Code.exe,115
7,84436c56-3b4b-4287-b605-6185395dadd0,2026-09-14,SnippingTool.exe,6
8,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,chrome.exe,161
9,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,Code.exe,121


In [2]:
import sys
sys.path.append('..')
from features import build_daily_dataframe, add_features,fill_missing_days, build_prediction_features

In [3]:
class SimpleApp:
    def __init__(self, process_name, duration_seconds):
        self.process_name = process_name
        self.duration_seconds = duration_seconds

class SimpleDay:
    def __init__(self, d, apps):
        self.date = d
        self.apps = apps

In [4]:
user_id = raw_df["UserId"].iloc[-1]
user_df = raw_df[raw_df["UserId"] == user_id]
user_df

,UserId,Date,ProcessName,DurationSeconds
2,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-14,chrome.exe,6
6,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-14,Code.exe,115
8,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,chrome.exe,161
9,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,Code.exe,121
10,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-15,datagrip64.exe,39
11,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-16,Code.exe,52
12,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-16,datagrip64.exe,36
13,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-16,chrome.exe,3
14,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-16,explorer.exe,4
15,01a0a63d-a9e7-742a-8f5a-cdea85b31ceb,2026-09-20,explorer.exe,16


In [5]:
history = []
for d, group in user_df.groupby("Date"):
    apps = [SimpleApp(row.ProcessName, row.DurationSeconds) for row in group.itertuples()]
    history.append(SimpleDay(d, apps))

len(history)

5

In [6]:
daily_df = build_daily_dataframe(history)
daily_df

,date,total_seconds,top_app
0,2026-09-14,121,Code.exe
1,2026-09-15,321,chrome.exe
2,2026-09-16,95,Code.exe
3,2026-09-20,11476,chrome.exe
4,2026-09-25,12258,chrome.exe


In [7]:
filled_df = fill_missing_days(daily_df)
filled_df

,date,total_seconds,top_app
0,2026-09-14,121.0,Code.exe
1,2026-09-15,321.0,chrome.exe
2,2026-09-16,95.0,Code.exe
3,2026-09-17,0.0,none
4,2026-09-18,0.0,none
5,2026-09-19,0.0,none
6,2026-09-20,11476.0,chrome.exe
7,2026-09-21,0.0,none
8,2026-09-22,0.0,none
9,2026-09-23,0.0,none


In [8]:
feature_df = add_features(filled_df)
feature_df

,date,total_seconds,top_app,day_of_week,is_weekend,previous_day_total,rolling_7day_avg
0,2026-09-14,121.0,Code.exe,0,0,NaN,NaN
1,2026-09-15,321.0,chrome.exe,1,0,121.0,121.000000
2,2026-09-16,95.0,Code.exe,2,0,321.0,221.000000
3,2026-09-17,0.0,none,3,0,95.0,179.000000
4,2026-09-18,0.0,none,4,0,0.0,134.250000
5,2026-09-19,0.0,none,5,1,0.0,107.400000
6,2026-09-20,11476.0,chrome.exe,6,1,0.0,89.500000
7,2026-09-21,0.0,none,0,0,11476.0,1716.142857
8,2026-09-22,0.0,none,1,0,0.0,1698.857143
9,2026-09-23,0.0,none,2,0,0.0,1653.000000


In [9]:
pred_features, tomorrow_date = build_prediction_features(filled_df)
print(tomorrow_date)
pred_features

2026-09-26 00:00:00


,day_of_week,is_weekend,previous_day_total,rolling_7day_avg
0,5,1,12258.0,3390.571429


In [10]:
feature_df = add_features(filled_df)

In [11]:
feature_df

,date,total_seconds,top_app,day_of_week,is_weekend,previous_day_total,rolling_7day_avg
0,2026-09-14,121.0,Code.exe,0,0,NaN,NaN
1,2026-09-15,321.0,chrome.exe,1,0,121.0,121.000000
2,2026-09-16,95.0,Code.exe,2,0,321.0,221.000000
3,2026-09-17,0.0,none,3,0,95.0,179.000000
4,2026-09-18,0.0,none,4,0,0.0,134.250000
5,2026-09-19,0.0,none,5,1,0.0,107.400000
6,2026-09-20,11476.0,chrome.exe,6,1,0.0,89.500000
7,2026-09-21,0.0,none,0,0,11476.0,1716.142857
8,2026-09-22,0.0,none,1,0,0.0,1698.857143
9,2026-09-23,0.0,none,2,0,0.0,1653.000000


In [ ]:
model_df = feature_df.dropna()

X = model_df[["day_of_week", "is_weekend", "previous_day_total", "rolling_7day_avg"]]
y = model_df["total_seconds"]

X